In [1]:
%run 00_functions.ipynb

# Daily Bitcoin Price Prediction

## 1. Load Trained Model and Scalers

In [2]:
# Define model directory
models_dir = Path("../models")

# Load trained LSTM model
final_model = load_model(
    models_dir / "lstm_model.keras"
)

# Load saved scalers
feature_scaler_final = joblib.load(
    models_dir / "feature_scaler.pkl"
)

target_scaler_final = joblib.load(
    models_dir / "target_scaler.pkl"
)

print("LSTM model and scalers loaded successfully.")

LSTM model and scalers loaded successfully.


In [3]:
# Preserve the original investing_info function
original_investing_info = historical_data.__globals__["investing_info"]

def investing_info_with_volume(investing_id):
    info = original_investing_info(investing_id)
    info["has_no_volume"] = False
    return info

historical_data.__globals__["investing_info"] = investing_info_with_volume

try:
    latest_data = historical_data(
        investing_id=1057391,
        from_date="01/01/2018",
        to_date=pd.Timestamp.now().strftime("%m/%d/%Y"),
        interval="D"
    )
finally:
    historical_data.__globals__["investing_info"] = original_investing_info

print("Available columns:", latest_data.keys())
print("Number of dates:", len(latest_data["date"]))
print("Number of volume values:", len(latest_data["volume"]))
print("Latest date:", latest_data["date"][-1])
print("Latest Close:", latest_data["close"][-1])

Available columns: dict_keys(['date', 'open', 'high', 'low', 'close', 'volume'])
Number of dates: 3205
Number of volume values: 3205
Latest date: 10/10/2026
Latest Close: 82860.8984375


In [ ]:
# Convert the collected data to a DataFrame
latest_df = pd.DataFrame(latest_data)

# Convert date to datetime
latest_df["date"] = pd.to_datetime(latest_df["date"])

# Sort by date and remove duplicate dates
latest_df = (
    latest_df
    .sort_values("date")
    .drop_duplicates(subset="date", keep="last")
    .reset_index(drop=True)
)

# Define the output path
data_path = Path("../data/processed/bitcoin_latest_data.csv")

# Save the latest data
latest_df.to_csv(data_path, index=False)

print("Latest Bitcoin data saved successfully.")
print("Data shape:", latest_df.shape)
print("Latest date:", latest_df["date"].max())
print("Latest Close:", latest_df["close"].iloc[-1])
print("Columns:", latest_df.columns.tolist())

## 2. Load Latest Bitcoin Data

In [3]:
# Load the latest Bitcoin data
data_path = Path("../data/processed/bitcoin_latest_data.csv")

latest_df = pd.read_csv(data_path)

# Convert date column to datetime
latest_df["date"] = pd.to_datetime(latest_df["date"])

print("Data shape:", latest_df.shape)
print("Latest date:", latest_df["date"].max())
print("Latest closing price:", latest_df["close"].iloc[-1])
print("Available columns:", latest_df.columns.tolist())

Data shape: (3205, 6)
Latest date: 2026-10-10 00:00:00
Latest closing price: 82860.78125
Available columns: ['date', 'open', 'high', 'low', 'close', 'volume']


## 3. Feature Engineering

In [4]:
# Exclude today's and future data before feature engineering
today = pd.Timestamp.now().normalize()

prediction_df = latest_df[
    latest_df["date"] < today
].copy()

print("Today's date:", today.date())
print("Latest input date:", prediction_df["date"].max())
print("Input rows:", len(prediction_df))

Today's date: 2026-10-10
Latest input date: 2026-10-09 00:00:00
Input rows: 3204


In [5]:
# Calculate MACD
ema_12 = prediction_df["close"].ewm(
    span=12,
    adjust=False
).mean()

ema_26 = prediction_df["close"].ewm(
    span=26,
    adjust=False
).mean()

prediction_df["MACD"] = ema_12 - ema_26


# Calculate RSI (14-day)
delta = prediction_df["close"].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.ewm(
    alpha=1/14,
    min_periods=14,
    adjust=False
).mean()

avg_loss = loss.ewm(
    alpha=1/14,
    min_periods=14,
    adjust=False
).mean()

rs = avg_gain / avg_loss

prediction_df["RSI"] = 100 - (100 / (1 + rs))


# Calculate SMA (20-day)
prediction_df["SMA"] = prediction_df["close"].rolling(
    window=20
).mean()


# Check the calculated features
print(prediction_df[
    ["date", "close", "RSI", "SMA", "MACD"]
].tail())

print("\nMissing values:")
print(prediction_df[["RSI", "SMA", "MACD"]].isna().sum())

           date         close        RSI           SMA         MACD
3199 2026-10-05  85771.007812  64.602503  83389.883203  1992.982994
3200 2026-10-06  85552.359375  63.436930  83857.930469  1934.503490
3201 2026-10-07  83322.140625  52.944086  84202.043359  1688.731402
3202 2026-10-08  81733.750000  46.983475  84244.628516  1350.220898
3203 2026-10-09  82623.601562  50.355490  84313.153906  1140.604292

Missing values:
RSI     14
SMA     19
MACD     0
dtype: int64


In [6]:
# Define the input features
feature_columns = [
    "open", "high", "low", "close",
    "volume", "RSI", "SMA", "MACD"
]

# Select the latest 90 days of data
latest_data = prediction_df[feature_columns].tail(90)

# Check the input data
print("Latest input date:", prediction_df["date"].iloc[-1])
print("Input shape:", latest_data.shape)
print("Missing values in latest 90 days:")
print(latest_data.isna().sum())

print("\nLast input date:", prediction_df["date"].iloc[-1])

Latest input date: 2026-10-09 00:00:00
Input shape: (90, 8)
Missing values in latest 90 days:
open      0
high      0
low       0
close     0
volume    0
RSI       0
SMA       0
MACD      0
dtype: int64

Last input date: 2026-10-09 00:00:00


In [7]:
# Set the look-back window
look_back = 90

print("Look-back window:", look_back)

Look-back window: 90


In [8]:
# Check whether 90 days of input data are available
if len(latest_data) != look_back:
    raise ValueError(
        f"Expected {look_back} input rows, "
        f"but received {len(latest_data)}."
    )

# Check for missing values
if latest_data.isnull().any().any():
    raise ValueError(
        "Missing values detected in the latest 90-day input data."
    )

print("Input data validation passed.")
print("Input rows:", len(latest_data))
print("Missing values:", latest_data.isnull().sum().sum())

Input data validation passed.
Input rows: 90
Missing values: 0


In [9]:
# Check whether the latest 90 input dates are consecutive
latest_dates = prediction_df["date"].tail(look_back)

date_gaps = latest_dates.diff().dropna()

if not (date_gaps == pd.Timedelta(days=1)).all():
    raise ValueError(
        "Date gaps detected in the latest 90-day input data."
    )

print("Date continuity check passed.")
print("First input date:", latest_dates.iloc[0].date())
print("Last input date:", latest_dates.iloc[-1].date())
print("Number of input days:", len(latest_dates))

Date continuity check passed.
First input date: 2026-07-12
Last input date: 2026-10-09
Number of input days: 90


In [10]:
# Scale the latest 90 days using the saved feature scaler
latest_data_scaled = feature_scaler_final.transform(
    latest_data[feature_columns]
)

# Reshape for LSTM: (samples, timesteps, features)
latest_sequence = latest_data_scaled.reshape(
    1, look_back, len(feature_columns)
)

print("Scaled input shape:", latest_data_scaled.shape)
print("LSTM input shape:", latest_sequence.shape)

Scaled input shape: (90, 8)
LSTM input shape: (1, 90, 8)


In [11]:
# Predict the next closing price
next_close_scaled = final_model.predict(
    latest_sequence,
    verbose=0
)

# Convert the prediction back to the actual BTC price
next_close = target_scaler_final.inverse_transform(
    next_close_scaled
)[0, 0]

# Get the latest input date and closing price
latest_input_date = prediction_df["date"].iloc[-1]
latest_close = prediction_df["close"].iloc[-1]

# Display prediction results
prediction_date = latest_input_date + pd.Timedelta(days=1)

print("Latest input date:", latest_input_date.date())
print(f"Latest actual Close: ${latest_close:,.2f}")
print("Prediction target date:", prediction_date.date())
print(f"Predicted next Close: ${next_close:,.2f}")

Latest input date: 2026-10-09
Latest actual Close: $82,623.60
Prediction target date: 2026-10-10
Predicted next Close: $83,626.52


## 4. Save Prediction Log

In [13]:

# Define prediction log path
log_path = Path("../data/processed/bitcoin_prediction_log.csv")

# Prepare actual closing prices
actual_price_map = latest_df.copy()
actual_price_map["date"] = pd.to_datetime(
    actual_price_map["date"]
).dt.normalize()

actual_price_map = (
    actual_price_map
    .drop_duplicates(subset="date", keep="last")
    .set_index("date")["close"]
)

# Load existing prediction log
if log_path.exists():
    prediction_log = pd.read_csv(log_path)

    if "predicted_close" in prediction_log.columns:
        prediction_log = prediction_log.rename(
            columns={"predicted_close": "prediction_price"}
        )
else:
    prediction_log = pd.DataFrame()

# Ensure required columns exist
required_columns = [
    "prediction_date",
    "prediction_price",
    "actual_price"
]

for column in required_columns:
    if column not in prediction_log.columns:
        prediction_log[column] = None

prediction_log = prediction_log[required_columns].copy()

# Convert data types
prediction_log["prediction_date"] = pd.to_datetime(
    prediction_log["prediction_date"],
    errors="coerce"
).dt.normalize()

prediction_log["prediction_price"] = pd.to_numeric(
    prediction_log["prediction_price"],
    errors="coerce"
)

prediction_log["actual_price"] = pd.to_numeric(
    prediction_log["actual_price"],
    errors="coerce"
)

# Remove invalid dates and duplicate prediction dates
prediction_log = (
    prediction_log
    .dropna(subset=["prediction_date"])
    .drop_duplicates(subset="prediction_date", keep="last")
    .reset_index(drop=True)
)

# Update actual prices when available
matched_actual_prices = prediction_log["prediction_date"].map(
    actual_price_map
)

prediction_log["actual_price"] = matched_actual_prices.combine_first(
    prediction_log["actual_price"]
)

# Add or replace today's prediction record
current_date = pd.Timestamp(prediction_date).normalize()

new_record = pd.DataFrame([{
    "prediction_date": current_date,
    "prediction_price": float(next_close),
    "actual_price": actual_price_map.get(current_date, None)
}])

prediction_log = prediction_log[
    prediction_log["prediction_date"] != current_date
]

prediction_log = pd.concat(
    [prediction_log, new_record],
    ignore_index=True
)

# Calculate prediction errors
prediction_log["absolute_error"] = (
    prediction_log["prediction_price"]
    - prediction_log["actual_price"]
).abs()

prediction_log["absolute_percentage_error"] = (
    prediction_log["absolute_error"]
    / prediction_log["actual_price"].abs()
    * 100
)

# Sort and save the combined log
prediction_log = prediction_log.sort_values(
    "prediction_date"
).reset_index(drop=True)

prediction_log.to_csv(log_path, index=False)

print("Prediction log and errors saved successfully.")
print("Total prediction records:", len(prediction_log))
print(prediction_log.tail())


Prediction log and errors saved successfully.
Total prediction records: 1
  prediction_date  prediction_price  actual_price  absolute_error  \
0      2026-10-10      83626.515625   82860.78125      765.734375   

   absolute_percentage_error  
0                   0.924122  


## 5. Prediction Error Evaluation

In [ ]:
# Calculate prediction errors
prediction_log["prediction_date"] = pd.to_datetime(
    prediction_log["prediction_date"]
)

prediction_log["absolute_error"] = (
    prediction_log["prediction_price"]
    - prediction_log["actual_price"]
).abs()

prediction_log["absolute_percentage_error"] = (
    prediction_log["absolute_error"]
    / prediction_log["actual_price"].abs()
    * 100
)

# Save the updated prediction log
prediction_log.to_csv(log_path, index=False)

print("Prediction log and errors saved successfully.")
print(
    prediction_log[
        [
            "prediction_date",
            "prediction_price",
            "actual_price",
            "absolute_error",
            "absolute_percentage_error"
        ]
    ].tail()
)

Prediction Error Summary:
  prediction_date  prediction_price  actual_price  absolute_error  \
0      2026-10-10      83626.515625   82860.78125      765.734375   

   absolute_percentage_error  
0                   0.924122  
